# NB6 — Step 2: extend training (resume from checkpoint-5000)

Your model stopped at 5000 steps (~0.63 epochs) and already scores **official
F1 0.4618** on QuranMB.v2 — matching the 2025 challenge winner's fine-tuned
checkpoint. You're undertrained; AraS2P ran ~2.5 epochs. This notebook **resumes
from checkpoint-5000** and trains more, with a lower decaying LR and early
stopping on dev PER (stops automatically when it plateaus — no wasted time).

It keeps the **last 2 checkpoints** so you can optionally average them later
(the free 'poor-man's ensemble' trick the top systems use). Output: a better
`final/` to re-score with NB5.

**Resume needs the FULL checkpoint** (optimizer.pt, scheduler.pt, trainer_state.json,
rng_state). If your saved checkpoint-5000 only has model weights, it falls back to
continuing from the weights with a fresh optimizer (still helps, just a soft restart).

In [ ]:
!pip install -q "transformers==4.44.2" "datasets==2.21.0" "accelerate==0.34.2" jiwer soundfile librosa
import warnings
warnings.filterwarnings("ignore", message=".*use_reentrant.*")
warnings.filterwarnings("ignore", category=UserWarning)

# PyTorch 2.6+ defaults torch.load to weights_only=True, which REFUSES to unpickle
# the checkpoint's rng_state.pth (it contains a numpy array) -> UnpicklingError on
# resume. Our own checkpoint is trusted, so patch torch.load back to weights_only=False.
import torch as _torch
_orig_load = _torch.load
def _patched_load(*a, **k):
    k.setdefault("weights_only", False)
    return _orig_load(*a, **k)
_torch.load = _patched_load
print("torch.load patched for resume (weights_only=False).")

In [ ]:
import json, os, time, shutil, glob
import numpy as np
import torch
from dataclasses import dataclass
from datasets import load_from_disk, Audio, concatenate_datasets

if torch.cuda.device_count() > 1:
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"
    print("Multiple GPUs -> restricting to GPU 0.")
print("visible GPUs:", torch.cuda.device_count())

MODEL_NAME = "facebook/w2v-bert-2.0"
SAMPLING_RATE = 16_000
PHONEME_COL = "phoneme_ref"
USE_TTS = True

CACHE_ROOT = "/kaggle/input/datasets/qasimmaajid/iqra-cache/iqra_prep"
LOCAL_TRAIN_DISK = f"{CACHE_ROOT}/iqra_train_arrow"
LOCAL_TTS_DISK   = f"{CACHE_ROOT}/iqra_tts_arrow"
VOCAB_PATH       = f"{CACHE_ROOT}/phoneme_vocab.json"

# ===========================================================================
# CHAINED TRAINING — run this notebook in SHORT chunks to dodge Kaggle's disk.
# Each chunk resumes from the previous chunk's saved checkpoint and adds ~3000
# steps (~2.5h, safe on disk). After each run: commit -> Create Dataset from the
# new checkpoint -> point RESUME_CKPT at it for the next chunk.
#
#   Chunk 1: RESUME_CKPT = .../checkpoint-5000   MAX_STEPS = 8000
#   Chunk 2: RESUME_CKPT = <chunk1 dataset>/checkpoint-8000   MAX_STEPS = 11000
#   Chunk 3: RESUME_CKPT = <chunk2 dataset>/checkpoint-11000  MAX_STEPS = 14000
#   ...stop when dev PER plateaus.
# ===========================================================================
RESUME_CKPT = "/kaggle/input/datasets/qasimmaajid/quran-phoneme-model/w2vbert-quran-phoneme/checkpoint-5000"   # EDIT each chunk
MAX_STEPS   = 8000     # EDIT each chunk (+~3000 over the resume point)
EVAL_STEPS  = 1000

OUTPUT_DIR  = "/kaggle/working/w2vbert-quran-phoneme-ext"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("resume ckpt exists:", os.path.isdir(RESUME_CKPT))
print("vocab exists:", os.path.exists(VOCAB_PATH))
print("this chunk -> MAX_STEPS:", MAX_STEPS)

In [ ]:
# Check what the checkpoint contains -> decide true resume vs soft restart.
ckpt_files = os.listdir(RESUME_CKPT) if os.path.isdir(RESUME_CKPT) else []
print("checkpoint contents:", ckpt_files)
has_optimizer = any(f.startswith("optimizer") for f in ckpt_files)
has_state     = "trainer_state.json" in ckpt_files
TRUE_RESUME = has_optimizer and has_state
print("optimizer.pt present:", has_optimizer, "| trainer_state.json:", has_state)
print("=> TRUE_RESUME" if TRUE_RESUME else "=> SOFT RESTART (continue from weights, fresh optimizer)")

In [ ]:
from transformers import AutoFeatureExtractor, Wav2Vec2CTCTokenizer
with open(VOCAB_PATH, encoding="utf-8") as f:
    vocab = json.load(f)
with open(os.path.join(OUTPUT_DIR, "vocab.json"), "w", encoding="utf-8") as f:
    json.dump(vocab, f, ensure_ascii=False)
tokenizer = Wav2Vec2CTCTokenizer(os.path.join(OUTPUT_DIR, "vocab.json"),
                                 unk_token="<unk>", pad_token="<pad>", word_delimiter_token=" ")
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)
print("vocab size:", len(vocab), "| blank id:", tokenizer.pad_token_id)

In [ ]:
# ---- Data (same as NB2: Iqra_train->phoneme_ref, TTS->phoneme_mis) ----
full = load_from_disk(LOCAL_TRAIN_DISK); train, dev = full["train"], full["dev"]

def normalise(ds, label_col):
    keep = {"audio", label_col}
    ds = ds.remove_columns([c for c in ds.column_names if c not in keep])
    if label_col != "labels_str": ds = ds.rename_column(label_col, "labels_str")
    return ds.cast_column("audio", Audio(sampling_rate=SAMPLING_RATE))

train = normalise(train, PHONEME_COL); dev = normalise(dev, PHONEME_COL)
if USE_TTS:
    tf = load_from_disk(LOCAL_TTS_DISK)
    tts = tf["train"] if "train" in getattr(tf, "keys", lambda: [])() else tf
    tcol = "phoneme_mis" if "phoneme_mis" in tts.column_names else PHONEME_COL
    tts = normalise(tts, tcol)
    train = concatenate_datasets([train, tts])
print("train:", len(train), "| dev:", len(dev))

In [ ]:
unk_id = tokenizer.convert_tokens_to_ids("<unk>")
def encode_labels(s): return [vocab.get(t, unk_id) for t in str(s).split() if t]
def make_transform(b):
    return {"input_features": [feature_extractor(a["array"], sampling_rate=SAMPLING_RATE).input_features[0] for a in b["audio"]],
            "labels": [encode_labels(s) for s in b["labels_str"]]}
train_proc = train.with_transform(make_transform)
dev_proc   = dev.with_transform(make_transform)

@dataclass
class CTCCollator:
    feature_extractor: object
    tokenizer: object
    def __call__(self, features):
        batch = self.feature_extractor.pad([{"input_features": f["input_features"]} for f in features], return_tensors="pt")
        labs = self.tokenizer.pad([{"input_ids": f["labels"]} for f in features], return_tensors="pt")
        batch["labels"] = labs["input_ids"].masked_fill(labs.attention_mask.ne(1), -100)
        return batch
collator = CTCCollator(feature_extractor, tokenizer)

import jiwer
id2tok = {v: k for k, v in vocab.items()}
def decode_ids(ids):
    out, prev, blank = [], None, vocab["<pad>"]
    for i in ids:
        if i != prev and i != blank: out.append(id2tok.get(int(i), "<unk>"))
        prev = i
    return " ".join(out)
def compute_metrics(pred):
    pi = np.argmax(pred.predictions, axis=-1)
    lab = np.where(pred.label_ids == -100, vocab["<pad>"], pred.label_ids)
    return {"per": jiwer.wer([decode_ids(l) for l in lab], [decode_ids(p) for p in pi])}
print("transforms + metrics ready")

In [ ]:
from transformers import Wav2Vec2BertForCTC, TrainingArguments, Trainer, EarlyStoppingCallback

model = Wav2Vec2BertForCTC.from_pretrained(
    RESUME_CKPT,
    vocab_size=len(vocab), pad_token_id=tokenizer.pad_token_id,
    ctc_loss_reduction="mean", add_adapter=True,
)
model.config.ctc_zero_infinity = True

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=8, per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_steps=200,
    max_steps=MAX_STEPS,
    fp16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    eval_strategy="steps", eval_steps=EVAL_STEPS, save_steps=EVAL_STEPS,
    # DISK: the prior runs died in write_ipynb (the notebook output log itself
    # bloats the .ipynb over hours) + checkpoint size. Mitigations: keep ONE
    # checkpoint, disable tqdm, log infrequently. Short chunks (8000) are the
    # real cure -- they finish before disk fills.
    save_total_limit=1,
    save_safetensors=True,
    logging_steps=500, disable_tqdm=True,
    load_best_model_at_end=True, metric_for_best_model="per", greater_is_better=False,
    dataloader_num_workers=2,
    remove_unused_columns=False, label_names=["labels"], report_to="none",
)

trainer = Trainer(model=model, args=args, train_dataset=train_proc, eval_dataset=dev_proc,
                  data_collator=collator, compute_metrics=compute_metrics,
                  callbacks=[EarlyStoppingCallback(early_stopping_patience=3)])
try: trainer.tokenizer = tokenizer
except Exception: pass
print("trainer ready | TRUE_RESUME =", TRUE_RESUME)

In [ ]:
# Resume + train. The Trainer writes checkpoint-XXXX folders every EVAL_STEPS to
# OUTPUT_DIR during training (save_total_limit=1 keeps the latest). Even if the
# FINAL save below dies on disk, that last checkpoint-XXXX is your resume point
# for the next chunk -> commit and Create Dataset from it regardless.
if TRUE_RESUME:
    print("Resuming full state from", RESUME_CKPT)
    trainer.train(resume_from_checkpoint=RESUME_CKPT)
else:
    print("Soft restart: continuing from checkpoint WEIGHTS, fresh optimizer.")
    trainer.train()

# Save the best model to a clean 'final' dir. Wrapped in try/except so a disk
# error here doesn't hide the fact that checkpoint-XXXX already exists.
try:
    trainer.save_model(os.path.join(OUTPUT_DIR, "final"))
    tokenizer.save_pretrained(os.path.join(OUTPUT_DIR, "final"))
    feature_extractor.save_pretrained(os.path.join(OUTPUT_DIR, "final"))
    print("saved final ->", os.path.join(OUTPUT_DIR, "final"))
except Exception as e:
    print("final save failed (disk?):", e)

# Always report what survived in OUTPUT_DIR for the next chunk.
survivors = [p for p in os.listdir(OUTPUT_DIR) if p.startswith("checkpoint") or p == "final"]
print("survivors in OUTPUT_DIR:", survivors)

### After this run
1. Commit, then **Create Dataset** from `w2vbert-quran-phoneme-ext/final`.
2. Re-run **NB5** with `MODEL_DIR` pointed at the new `final` -> get the new official F1.
3. If F1 went up, great. If dev PER barely moved, you've hit the data ceiling and
   the next gains come from Step 3 (confidence gating) and Step 4 (real-error data),
   not more epochs.

**Optional free ensemble:** keep both `checkpoint-*` folders this run saved; later
we can average their weights (a cheap +1-2 F1 trick the top systems use).